In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

def build_gold_fato_transacao():
    print("Construindo a gold_fato_transacao...")
    
    query = """
    CREATE OR REPLACE TABLE workspace.default.gold_fato_transacao AS
    
    -- 1. CTE de Transações base
    WITH transacoes AS (
        SELECT * FROM workspace.default.silver_transacoes
    ),
    
    -- 2. CTE de Estornos (Preparação para anular o valor líquido)
    estornos AS (
        SELECT id_transacao, data_estorno, motivo 
        FROM workspace.default.silver_estornos
    ),
    
    -- 3. CTE do Histórico de Cartões (Tratando valid_to nulo como o "infinito")
    cartoes_hist AS (
        SELECT 
            id_cartao, 
            id_conta, 
            tipo_cartao, 
            status_cartao,
            valid_from, 
            COALESCE(valid_to, '9999-12-31 23:59:59') AS valid_to
        FROM workspace.default.silver_cartoes
    ),
    
    -- 4. CTE Final integrando as regras de negócio
    fato_enriquecida AS (
        SELECT 
            t.id_transacao,
            t.id_cartao,
            c.id_conta,
            t.data_transacao,
            t.mcc,
            t.estabelecimento,
            t.canal,
            
            -- Regra de Negócio: Se tem estorno, o valor líquido é zerado
            t.valor AS valor_bruto,
            CASE WHEN e.id_transacao IS NOT NULL THEN 0 ELSE t.valor END AS valor_liquido,
            
            -- Flags analíticas
            CASE WHEN e.id_transacao IS NOT NULL THEN TRUE ELSE FALSE END AS is_estornada,
            e.motivo AS motivo_estorno,
            
            -- Snapshot do status do cartão no momento da compra
            c.status_cartao AS status_cartao_no_momento
            
        FROM transacoes t
        
        -- Join com Estornos
        LEFT JOIN estornos e 
            ON t.id_transacao = e.id_transacao
            
        -- POINT-IN-TIME JOIN (SCD2): Pega a versão exata do cartão na data da transação
        LEFT JOIN cartoes_hist c 
            ON t.id_cartao = c.id_cartao 
            AND t.data_transacao >= c.valid_from 
            AND t.data_transacao < c.valid_to
    )
    
    SELECT * FROM fato_enriquecida;
    """
    
    spark.sql(query)
    print("Sucesso! Tabela workspace.default.gold_fato_transacao criada.")

# Executa a criação da fato
build_gold_fato_transacao()

# Exibe uma amostra para validarmos se o valor_liquido zerou nos estornos
display(spark.sql("""
    SELECT id_transacao, data_transacao, valor_bruto, valor_liquido, is_estornada, status_cartao_no_momento
    FROM workspace.default.gold_fato_transacao 
    ORDER BY is_estornada DESC 
    LIMIT 5
"""))